# AI 기반 OODA Loop 전처리 에이전트

이 노트북은 이미지의 상태를 진단하고 LLM(GPT-4o)의 판단을 통해 최적의 보정 도구를 선택하는 지능형 OODA Loop 시스템을 구현합니다.

## 주요 프로세스

1.  **Metric 추출 (Measure)**: 이미지의 밝기, 대비, 선명도 수치화
2.  **AI 판단 (Orient & Decide)**: 수치와 문맥을 고려하여 AI가 다음 행동 결정
3.  **도구 실행 (Act)**: 선택된 보정 필터 적용
4.  **재검증 (Loop)**: 개선 여부 확인 후 반복 또는 종료

In [ ]:
# 1. 환경 설정 및 라이브러리 임포트
import sys
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import operator
from typing import TypedDict, List, Optional, Dict, Any, Literal, Annotated
from pathlib import Path

# LangChain / LangGraph
from langchain_core.pydantic_v1 import BaseModel, Field
# from langchain_openai import ChatOpenAI  <-- 제거
from src.agents.gemini_chatmodel import GeminiChatModel # <-- Project AI Model
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END

# 프로젝트 루트 경로 설정
project_root = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(project_root))

# 환경 변수 로드 (OpenAI API Key 등)
from dotenv import load_dotenv
load_dotenv(project_root / ".env")

print("환경 설정 완료")

In [ ]:
# 2. LangGraph State 정의

class OODAState(TypedDict):
    original_image: np.ndarray       # 원본 이미지
    current_image: np.ndarray        # 현재 처리 중인 이미지
    metrics: dict                    # 현재 이미지의 상태 수치
    # Annotated[List, operator.add]를 사용하여 리스트가 덮어씌워지지 않고 누적되도록 설정
    history: Annotated[List[str], operator.add]   # 적용한 도구 기록
    loop_count: int                  # 반복 횟수

print("State 정의 완료")

In [ ]:
# 3. 노드 정의 A: 측정 (Measure/Observe)

def calculate_metrics(image: np.ndarray) -> dict:
    """이미지 품질 지표 계산"""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    total_pixels = h * w

    # 히스토그램 분석
    hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
    shadow_ratio = float(np.sum(hist[:50]) / total_pixels)     # 암부 비율
    highlight_ratio = float(np.sum(hist[200:]) / total_pixels) # 명부 비율

    metrics = {
        "shadow_ratio": round(shadow_ratio, 3),
        "highlight_ratio": round(highlight_ratio, 3),
        "contrast_score": round(float(gray.std()), 1),
        "sharpness_score": round(float(cv2.Laplacian(gray, cv2.CV_64F).var()), 1)
    }
    return metrics

def node_observe(state: OODAState) -> dict:
    """Observe Node: 이미지 상태 측정"""
    print("\n🔍 [Observe] Measuring Metrics...")
    metrics = calculate_metrics(state["current_image"])
    return {"metrics": metrics}

print("Observe Node 정의 완료")

In [ ]:
# 4. 노드 정의 B: 실행 (Act)

def apply_gamma(image: np.ndarray, gamma=1.5) -> np.ndarray:
    inv_gamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv_gamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
    return cv2.LUT(image, table)

def apply_clahe(image: np.ndarray) -> np.ndarray:
    img_yuv = cv2.cvtColor(image, cv2.COLOR_BGR2YUV)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
    img_yuv[:,:,0] = clahe.apply(img_yuv[:,:,0])
    return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)

def apply_sharpen(image: np.ndarray) -> np.ndarray:
    kernel = np.array([[-1,-1,-1], [-1,9,-1], [-1,-1,-1]])
    return cv2.filter2D(image, -1, kernel)

# Wrappers
def node_act_gamma(state: OODAState) -> dict:
    print("🛠️ [Act] Applying Gamma Correction...")
    new_img = apply_gamma(state["current_image"])
    return {
        "current_image": new_img, 
        "history": ["act_gamma"], 
        "loop_count": state["loop_count"] + 1
    }

def node_act_clahe(state: OODAState) -> dict:
    print("🛠️ [Act] Applying CLAHE...")
    new_img = apply_clahe(state["current_image"])
    return {
        "current_image": new_img, 
        "history": ["act_clahe"], 
        "loop_count": state["loop_count"] + 1
    }

def node_act_sharpen(state: OODAState) -> dict:
    print("🛠️ [Act] Applying Sharpening...")
    new_img = apply_sharpen(state["current_image"])
    return {
        "current_image": new_img, 
        "history": ["act_sharpen"], 
        "loop_count": state["loop_count"] + 1
    }

print("Action Nodes 정의 완료")

In [ ]:
# 5. AI 판단 로직 설정 (Orient/Decide)

# 5.1 출력 구조체 정의
class RoutingDecision(BaseModel):
    """다음 단계에 실행할 도구를 결정합니다."""
    next_action: Literal["act_gamma", "act_clahe", "act_sharpen", "done"] = Field(
        ..., 
        description="이미지 상태(Metric)에 따라 실행할 최적의 도구 또는 종료(done)를 선택"
    )
    reasoning: str = Field(
        ..., 
        description="왜 이 도구를 선택했는지에 대한 짧은 근거 (예: 암부 비율이 너무 높음)"
    )

# 5.2 LLM 및 프롬프트 설정
# [변경] 프로젝트 표준 모델인 GeminiChatModel 사용
llm = GeminiChatModel(temperature=0)
# Note: GeminiChatModel이 with_structured_output을 지원하는지 확인 필요.
# 만약 지원하지 않는다면 단순 invoke 후 파싱하는 로직이 필요할 수 있으나,
# LangChain 호환성을 위해 우선 시도합니다.
structured_llm = llm.with_structured_output(RoutingDecision)

# 2. 판단용 시스템 프롬프트
system_prompt = """
당신은 화재조사 이미지 전처리 전문가 AI입니다.
현재 이미지의 분석 수치(Metric)와 이전 행동(History)을 보고, **가장 시급한 문제를 해결할 도구 하나**를 선택하십시오.

**[분석 기준 가이드]**
1. **Shadow Ratio (암부 비율)**: 
   - 0.60 이상이면 매우 어두움 -> `act_gamma` 추천.
   - 0.60 미만이면 양호.
2. **Highlight Ratio (명부 비율)**:
   - 0.30 이상이면 반사가 심함 -> `act_clahe` 추천.
3. **Contrast (대비)**:
   - 40.0 미만이면 뿌옇고 흐릿함 -> `act_clahe` 추천.
4. **Sharpness (선명도)**:
   - 100.0 미만이면 경계가 불분명함 -> `act_sharpen` 추천.

**[제약 사항]**
- 이미 사용한 도구(`history`에 있는 도구)는 다시 선택하지 마십시오.
- 모든 수치가 양호하거나, 이미 필요한 조치를 다 했다면 `done`을 선택하십시오.
- `act_gamma`와 `act_clahe` 중 고민된다면, 안 보이는 걸 보이게 하는 `act_gamma`가 우선입니다.
"""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Current Metrics: {metrics}\nTool History: {history}")
])

# 3. 체인 생성 (Prompt -> LLM -> Pydantic Parsing)
decision_chain = prompt_template | structured_llm

# 3. LangGraph의 Conditional Edge 함수 수정
def decide_next_step_with_ai(state: OODAState) -> Literal["act_gamma", "act_clahe", "act_sharpen", "done"]:
    """
    [Orient & Decide] 
    LLM이 Metrics를 보고 다음 행동을 결정합니다.
    """
    metrics = state["metrics"]
    history = state["history"]
    loop_count = state["loop_count"]

    # 1. 강제 종료 조건 (비용 절약 및 루프 방지)
    if loop_count >= 3:
        print(" -> [AI Decide] Max loop count reached. Forcing stop.")
        return "done"

    # 2. AI에게 판단 요청
    print(f" -> [AI Thinking] Metrics: {metrics}, History: {history}")

    try:
        # LLM 호출
        decision: RoutingDecision = decision_chain.invoke({
            "metrics": metrics,
            "history": history
        })

        print(f" -> [AI Decision] Choice: {decision.next_action} / Reason: {decision.reasoning}")
        return decision.next_action
    except Exception as e:
        print(f"❌ [AI Error] {e}. Fallback to done.")
        return "done"

print("AI Decision Logic 정의 완료 (Gemini Model)")

In [ ]:
# 6. 그래프 빌드

def build_ai_ooda_graph():
    workflow = StateGraph(OODAState)

    # 노드 추가 (기존과 동일)
    workflow.add_node("observe", node_observe)
    workflow.add_node("act_gamma", node_act_gamma)
    workflow.add_node("act_clahe", node_act_clahe)
    workflow.add_node("act_sharpen", node_act_sharpen)

    workflow.set_entry_point("observe")

    # [핵심 변경] 판단 로직을 AI 함수로 교체
    workflow.add_conditional_edges(
        "observe",
        decide_next_step_with_ai,  # <--- 여기가 AI 함수로 변경됨
        {
            "act_gamma": "act_gamma",
            "act_clahe": "act_clahe",
            "act_sharpen": "act_sharpen",
            "done": END
        }
    )

    workflow.add_edge("act_gamma", "observe")
    workflow.add_edge("act_clahe", "observe")
    workflow.add_edge("act_sharpen", "observe")

    return workflow.compile()

print("Workflow Build 완료")

In [ ]:
# 7. 실행 테스트
from src.utils import find_data_directory

data_dir = find_data_directory()
test_image_name = "IMG_8113.jpg"  # 테스트하고 싶은 이미지
test_image_path = Path(data_dir) / test_image_name

if not test_image_path.exists():
    # 없으면 아무 jpg나 잡기
    images = list(Path(data_dir).glob("*.jpg"))
    if images:
        test_image_path = images[0]
    else:
        print("❌ 테스트 이미지가 없습니다.")
        test_image_path = None

if test_image_path:
    print(f"📂 Loading image: {test_image_path.name}")
    original_img = cv2.imread(str(test_image_path))
    
    if original_img is not None:
        app = build_ai_ooda_graph()
        
        input_state: OODAState = {
            "original_image": original_img,
            "current_image": original_img.copy(),
            "metrics": {},
            "history": [],
            "loop_count": 0
        }
        
        print("🚀 Workflow Start...")
        final_state = app.invoke(input_state)
        print("✨ Workflow Finished!")
        
        # 결과 시각화
        plt.figure(figsize=(14, 6))
        plt.subplot(1, 2, 1)
        plt.title("Original")
        plt.imshow(cv2.cvtColor(final_state['original_image'], cv2.COLOR_BGR2RGB))
        plt.axis('off')
        
        plt.subplot(1, 2, 2)
        plt.title(f"Optimized (History: {final_state['history']})")
        plt.imshow(cv2.cvtColor(final_state['current_image'], cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.show()
    else:
        print("❌ 이미지 로드 실패")